# Production Feature Engineering Pipeline
## Home Credit Default Risk

This notebook demonstrates the frozen, reusable feature engineering pipeline used to construct the production 166-feature model representation from the raw Home Credit tables.

### Architecture & Scope
- **Research Closed**: Iterative feature exploration, ablation studies, and recursive feature elimination (RFE) are complete.
- **Frozen Schema**: Exactly 166 features (`ACCEPTED_FINAL_FEATURES`) defined in `home_credit.schema`.
- **Reusable Implementation**: All feature logic is packaged under `src/home_credit/features/` and directly reusable for batch training and real-time/API inference.
- **Strict Leakage Prevention**: Cutoff audits enforce that only records known at or prior to loan decision are utilized.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

# Reusable feature engineering package
from home_credit.features import (
    build_application_features,
    build_bureau_features,
    build_bureau_balance_features,
    build_previous_application_features,
    build_credit_card_features,
    build_installments_features,
    build_pos_cash_features,
    assemble_features,
)
from home_credit.schema import (
    ACCEPTED_FINAL_FEATURES,
    ACCEPTED_CATEGORICAL_FEATURES,
)

# Repository-relative data path resolution
DATA_DIR = Path("data/raw") if Path("data/raw").exists() else Path("../data/raw")
print(f"Data directory: {DATA_DIR.resolve()}")
print(f"Target schema: {len(ACCEPTED_FINAL_FEATURES)} features ({len(ACCEPTED_CATEGORICAL_FEATURES)} categorical)")


### 1. Raw Data Sources Overview
Home Credit provides a relational schema spanning applicant demographic data and multiple historical credit/repayment tables.


In [ ]:
raw_files = [
    "application_train.csv",
    "bureau.csv",
    "bureau_balance.csv",
    "previous_application.csv",
    "installments_payments.csv",
    "credit_card_balance.csv",
    "POS_CASH_balance.csv",
]

table_info = []
for fname in raw_files:
    fpath = DATA_DIR / fname
    if fpath.exists():
        size_mb = fpath.stat().st_size / (1024 * 1024)
        table_info.append({"Table": fname, "Status": "Found", "Size (MB)": f"{size_mb:.1f}"})
    else:
        table_info.append({"Table": fname, "Status": "Missing", "Size (MB)": "N/A"})

pd.DataFrame(table_info)


### 2. Application Preprocessing & Base Feature Extraction
#### Accepted Application Feature Design
The application dataset captures baseline applicant characteristics at loan origination:
- **Demographic & Employment**: `DAYS_BIRTH` and `DAYS_EMPLOYED` are normalized to years (`AGE_YEARS = -DAYS_BIRTH / 365.25`, `EMPLOYED_YEARS = -DAYS_EMPLOYED / 365.25`).
- **Sentinel Anomaly Treatment**: `DAYS_EMPLOYED = 365243` represents an impossible positive value (unemployment / retired / non-wage earner). It is replaced with `np.nan` while preserving the anomaly signal through `DAYS_EMPLOYED_ANOMALY = 1`.
- **Financial Ratios**: Added in early milestone `E2`, these three burden ratios provided significant signal (+0.0064 ROC-AUC over raw baseline):
  - `CREDIT_INCOME_RATIO = AMT_CREDIT / AMT_INCOME_TOTAL`: Measures total leverage relative to annual income.
  - `ANNUITY_INCOME_RATIO = AMT_ANNUITY / AMT_INCOME_TOTAL`: Measures payment burden against cash flow.
  - `ANNUITY_CREDIT_RATIO = AMT_ANNUITY / AMT_CREDIT`: Proxy for loan term duration and implicit interest rate.
- **Housing Quality & Missingness**: Aggregated missingness rate across all building / apartment features (`HOUSING_INFO_MISSING_PCT`) captures socio-economic signal without overfitting individual building attributes.
- **Categorical Missing Representation**: Missing values in the 15 categorical features are explicitly labeled as `"__MISSING__"` and converted to pandas `category` dtypes.


In [ ]:
# Sample application data demonstration
app_raw = pd.read_csv(DATA_DIR / "application_train.csv", nrows=1000)
app_features = build_application_features(app_raw)
print(f"Application raw shape: {app_raw.shape} -> Processed shape: {app_features.shape}")
app_features[["SK_ID_CURR", "CREDIT_INCOME_RATIO", "ANNUITY_INCOME_RATIO", "ANNUITY_CREDIT_RATIO", "AGE_YEARS", "EMPLOYED_YEARS"]].head()


### 3. Bureau Table Features (`build_bureau_features`)
#### Accepted Logic & Cutoff Rules
Extracts external credit history from the Credit Bureau:
- **Cutoff Audit**: `DAYS_CREDIT_UPDATE <= 0` filters out post-application record leaks.
- **Credit Status & Volume**: Active, closed, and sold credit counts, plus `BUREAU_ACTIVE_SHARE = ACTIVE_COUNT / TOTAL_COUNT`.
- **Recency Windows**: Credits opened within the last 180, 365, and 730 days.
- **Exposure & Overdue**: Total debt, max credit sum, total credit limit, and historical max overdue (`MAX_CREDIT_OVERDUE_AMT`).
- **Accepted Schema**: Exactly 16 bureau features in `ACCEPTED_FINAL_FEATURES`.


In [ ]:
bureau_raw = pd.read_csv(DATA_DIR / "bureau.csv", nrows=5000)
bureau_feats = build_bureau_features(bureau_raw)
print(f"Bureau features shape: {bureau_feats.shape}")
assert bureau_feats["SK_ID_CURR"].is_unique, "SK_ID_CURR must be unique"
bureau_feats.head()


### 4. Previous Applications Features (`build_previous_application_features`)
#### Accepted Logic & Ratios
Extracts internal Home Credit prior loan application history:
- **Application vs Credit Dynamics**: `CREDIT_APPLICATION_DIFF` and `CREDIT_APPLICATION_RATIO` measure credit granting conservatism.
- **Temporal Sentinel Auditing**: Sentinel values (`365243`) in due/termination dates are replaced with `np.nan`.
- **Contract Durations & Future Commitments**: Planned duration, future termination counts, and maximum days remaining.
- **Accepted Schema**: Exactly 34 previous application features in `ACCEPTED_FINAL_FEATURES`.


In [ ]:
prev_raw = pd.read_csv(DATA_DIR / "previous_application.csv", nrows=5000)
prev_feats = build_previous_application_features(prev_raw)
print(f"Previous application features shape: {prev_feats.shape}")
assert prev_feats["SK_ID_CURR"].is_unique, "SK_ID_CURR must be unique"
prev_feats.head()


### 5. Credit Card Features (`build_credit_card_features`)
#### Accepted Logic & Utilization
Extracts credit card balance and utilization behavior:
- **Account Activity**: Active balance month share and drawing month share.
- **Utilization Ratios**: Mean, max, and latest utilization (`AMT_BALANCE / AMT_CREDIT_LIMIT_ACTUAL`).
- **Latest State**: Tracks the final observed month per contract before loan application.
- **Accepted Schema**: Exactly 12 credit card features in `ACCEPTED_FINAL_FEATURES`.


In [ ]:
cc_raw = pd.read_csv(DATA_DIR / "credit_card_balance.csv", nrows=5000)
cc_feats = build_credit_card_features(cc_raw)
print(f"Credit card features shape: {cc_feats.shape}")
assert cc_feats["SK_ID_CURR"].is_unique, "SK_ID_CURR must be unique"
cc_feats.head()


### 6. Installments Payments Features (`build_installments_features`)
#### Accepted Logic (IP1, IP2, IP3)
Extracts repayment discipline and shortfall tracking:
- **Installment-Level Aggregation**: Groups fragmented payments into single scheduled installment events.
- **Lateness & Shortfalls**: Delay in days, underpaid share, shortfall amount, and coverage ratio (`AMT_PAYMENT / AMT_INSTALMENT`).
- **Temporal Windows**: Aggregates recent 6-month and 12-month repayment performance to capture recent stress.
- **F8 / IPX Rejection**: Complex two-stage contract-level dynamics were rejected in research (-0.00030 ΔAP) and are strictly excluded from production.
- **Accepted Schema**: Exactly 19 installments features in `ACCEPTED_FINAL_FEATURES`.


In [ ]:
ip_raw = pd.read_csv(DATA_DIR / "installments_payments.csv", nrows=5000)
ip_feats = build_installments_features(ip_raw)
print(f"Installments features shape: {ip_feats.shape}")
assert ip_feats["SK_ID_CURR"].is_unique, "SK_ID_CURR must be unique"
ip_feats.head()


### 7. POS / Cash Balance Features (`build_pos_cash_features`)
#### Accepted F9 Dynamic Features (POSX)
Extracts dynamic contract completion and delinquency trajectory:
- **Contract Progress**: Progress ratio comparing oldest remaining installment ratio to latest remaining installment ratio.
- **Recent Deterioration**: `POSX_MEAN_RECENT_WORSENING` and `POSX_MAX_RECENT_WORSENING` measure delinquency escalation over the last 6 months.
- **Latest DPD**: Status flags for delinquency at the most recent observed point.
- **Accepted Schema**: Exactly 9 POSX features in `ACCEPTED_FINAL_FEATURES`.


In [ ]:
pos_raw = pd.read_csv(DATA_DIR / "POS_CASH_balance.csv", nrows=5000)
pos_feats = build_pos_cash_features(pos_raw)
print(f"POS/Cash features shape: {pos_feats.shape}")
assert pos_feats["SK_ID_CURR"].is_unique, "SK_ID_CURR must be unique"
pos_feats.head()


### 8. Bureau Balance Dynamic Features (`build_bureau_balance_features`)
#### Accepted F9 Dynamic Features (BBX)
Extracts monthly bureau delinquency transitions:
- **Severity Mapping**: Maps status codes ('C','0' -> 0, '1'-'5' -> numeric severity, 'X' -> NaN).
- **Recent Severity & Delinquency**: 6-month and 12-month delinquency share, maximum severity, and deterioration rates.
- **Accepted Schema**: Exactly 9 BBX features in `ACCEPTED_FINAL_FEATURES`.


In [ ]:
bb_raw = pd.read_csv(DATA_DIR / "bureau_balance.csv", nrows=10000)
bureau_link = bureau_raw[["SK_ID_BUREAU", "SK_ID_CURR"]].drop_duplicates()
bb_feats = build_bureau_balance_features(bb_raw, bureau=bureau_link)
print(f"Bureau balance features shape: {bb_feats.shape}")
assert bb_feats["SK_ID_CURR"].is_unique, "SK_ID_CURR must be unique"
bb_feats.head()


### 9. Final Feature Matrix Assembly & Schema Validation
All feature tables are merged with `application` via left joins, enforcing one-to-one joins and standard missingness semantics.


In [ ]:
assembled = assemble_features(
    application=app_raw,
    bureau_features_df=bureau_feats,
    bureau_balance_features_df=bb_feats,
    previous_application_features_df=prev_feats,
    credit_card_features_df=cc_feats,
    installments_features_df=ip_feats,
    pos_cash_features_df=pos_feats,
    include_id=True,
)

print(f"Assembled matrix shape: {assembled.shape}")

# Visible assertions
assert assembled["SK_ID_CURR"].is_unique, "Applicant IDs must be unique"
assert len(ACCEPTED_FINAL_FEATURES) == 166, "Schema must have exactly 166 features"
assert not pd.Index(ACCEPTED_FINAL_FEATURES).duplicated().any(), "No duplicate feature names allowed"

# Model feature view
X_model = assembled[list(ACCEPTED_FINAL_FEATURES)].copy()
assert X_model.shape[1] == 166, f"Expected 166 columns, got {X_model.shape[1]}"
assert list(X_model.columns) == list(ACCEPTED_FINAL_FEATURES), "Column order must match ACCEPTED_FINAL_FEATURES"

num_categorical = sum(isinstance(X_model[col].dtype, pd.CategoricalDtype) for col in X_model.columns)
total_missing = X_model.isna().sum().sum()
total_cells = X_model.shape[0] * X_model.shape[1]

print("--- Feature Matrix Summary ---")
print(f"Rows: {X_model.shape[0]}")
print(f"Model features: {X_model.shape[1]}")
print(f"Categorical features: {num_categorical}")
print(f"Missing cells: {total_missing} / {total_cells} ({total_missing / total_cells * 100:.2f}%)")


### Conclusion & Pipeline Reuse
The reusable feature pipeline successfully reproduces the frozen **166-feature** schema from raw Home Credit tables.
- Guarantees exact column ordering and categorical dtypes expected by the trained CatBoost, LightGBM, and XGBoost models.
- The shared module `home_credit.features.assemble` and `home_credit.features.application` will be directly imported by future FastAPI inference services.
